# DDPM (Denoising Diffusion Probabilistic Models) part 2 - DDPM 实现

这个模型我将按照**我的理解**来进行整理与介绍，DDPM 的示例加噪和去噪过程如下图所示:

<div align="center">
    <img src="./imgs/DDPM_example.png" alt="DDPM_example" style="width: 750px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/563661713)

[DDPM 论文](https://arxiv.org/pdf/2006.11239)

[DDPM Pytorch 实现](https://github.com/labmlai/annotated_deep_learning_paper_implementations/tree/master/labml_nn/diffusion/ddpm)

## DDPM 原理
在这个部分，我们就要正式开始 DDPM 模型的讲解了，在本篇中，我会尽量忽略复杂的推导，力求**直白**地展示出 DDPM 的原理和过程

In [5]:
from typing import Tuple, Optional
import torch
import torch.nn.functional as F
import torch.utils.data
from torch import nn
from c_ddpm import vis_img_change
from PIL import Image
from torchvision.utils import save_image
import os


### 前向过程（forward process）和反向过程（reverse process）
扩散模型包括向前加噪和反向去噪的过程，加噪的扩散过程就是对原始图像$x_0$不断地一步一步进行加噪，t步后最终得到噪声数据$x_t$；反向过程就是加噪的逆过程，即从$x_t$不断地一步一步进行去噪，得到原始图像$x_0$，我们的最终目标就是让模型能够从原始噪声中生成图像


#### 前向过程（forward process）
在向前过程(扩散过程)中，我们一步一步地对原始图像添加高斯噪声，直到图像变成随机的噪音，对于原始数据$x_0 \sim q(x_0)$($q_{x_0}$是真实的图像分布，这里你可以理解成，从训练的真实图像中抽取图像)，T步的扩散过程的每一步都是对上一步的数据$x_{t-1}$按照下面的方式增加高斯噪声:

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\left(x_t; \sqrt{1-\beta_t}x_{t-1}, \beta_t \mathbf{I}\right)
$$

意思是说，在已知上一步加噪结果$x_{t-1}$的情况下，$x_t$的分布是从均值为$\sqrt{1-\beta_t}x_{t-1}$，方差(协方差矩阵)为$\beta_t \mathbf{I}$的高斯分布中采样得到的；因为这一步的结果只依赖于上一步，所以就是一个**马尔科夫链**:

$$
q(x_{1:T} \mid x_0) = \prod_{t=1}^T q(x_t \mid x_{t-1})
$$

在具体实现中，使用了重参数化技巧，就是说能够直接从$x_0$计算加噪t步后的结果$x_t$，公式如下:

`function: q_sample:`
$$
x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon
$$

反重参数化可得(给定$x_0$时$x_t$的分布):

`function: q_xt_x0:`
$$
q(x_t \mid x_0) = \mathcal{N}\left(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) \mathbf{I}\right)
$$

其中:

$$
\begin{aligned}
&\epsilon \sim \mathcal{N}(0, \mathbf{I}): \text{ 标准高斯噪声} \\[4pt]
&\bar{\alpha}_t = \prod_{s=1}^t \alpha_s,\quad \alpha_t = 1 - \beta_t
\end{aligned}
$$

这里介绍一下$\beta$，它是噪声调度，比如$\beta_1 = 0.0001, \beta_T = 0.02$，中间值通过线性插值得到，它的作用是让每一步只添加微量的噪声，从而让图像变化更加平滑


In [2]:
def q_sample(x0: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """
    Get xt from x0 after noising t steps.
    这个函数用于演示加噪和去噪的过程，具体实现以最终的实现类为主
    """
    # 噪声调度的初值、末值、加噪总步数
    beta_start = 0.0001
    beta_end = 0.02
    n_steps = 1000
    # 噪声调度
    beta = torch.linspace(beta_start, beta_end, n_steps)
    alpha = 1.0 - beta  # \alpha_t
    alpha_bar = torch.cumprod(alpha, dim=0)  # \bar{\alpha_t}，即连乘

    eps = torch.randn_like(x0)  # 标准高斯噪声
    # 单个\sqrt{\bar{\alpha_t}}值 -> [bs, 1, 1 ,1] -> [bs, C, H, W](运算时广播)
    sqrt_alpha_bar = torch.sqrt(alpha_bar[t]).view(-1, 1, 1, 1)  
    # 单个 1 - \sqrt{\bar{\alpha_t}} 值 -> [bs, 1, 1 ,1] -> [bs, C, H, W](运算时广播)
    sqrt_one_minus_alpha_bar = torch.sqrt(1 - alpha_bar[t]).view(-1, 1, 1, 1)

    xt = sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * eps

    return xt
    

vis_img_change(origin_img_path='./imgs/gugugaga.png',
               vis_interval=50, 
               func=q_sample,
               output_dir='./imgs',
               gif_name='gugugaga_noising.gif') 


GIF saved to: imgs\gugugaga_noising.gif


可以看看程序得到的gif图:

<div align="center">
    <img src="./imgs/gugugaga_noising.gif" alt="gugugaga_noising" style="width: 250px; height: auto;">
</div>


#### 反向过程（reverse process）
反向过程就是去噪的过程，如果我们知道每一步添加的噪声，那么就可以从$x_t开始一步一步减去噪声，得到原始的图像$x_0$，那么我们就需要估计分布$q(x_{t-1} \mid x_t)$，这个就是用一个神经网络来估计，网络参数抽象成$\theta$:

$$
p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^T p_\theta(x_{t-1} \mid x_t)
$$

`function p_sample:`
$$
p_\theta(x_{t-1} \mid x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))
$$

我们理解一下上面两个式子:
- 式子1意思是说，我们从噪声中采样$x_t$，用神经网络一步一步预测噪声，就得到了预测的原始图像$x_0$；

- 式子2意思是说，已知$x_t$的情况下，$x_{t-1}$服从均值为$\mu_\theta(x_t, t)$，方差为$\Sigma_\theta(x_t, t)$的高斯分布，均值和方差是神经网络根据当前的$x_t$和加噪步t预测得到的

在论文中，其实模型预测的是第t步添加的噪声$\epsilon_t$，即$\epsilon_\theta(x_t, t)$，然后通过下面的公式得到均值和方差，最后再从分布中采样得到$x_{t-1}$:

$$
\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} (x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha_t}}} \epsilon_\theta(x_t, t))

\\

\sigma^2 = \beta_t
$$

然后得到高斯分布后，采样，就得到了$x_{t-1}$了

注意，模型推理时，是需要从纯噪声**一步一步**去噪得到原始图像的


In [3]:
# 生成用于展示的去噪GIF
vis_img_change(origin_img_path='./imgs/gugugaga.png',
               vis_interval=50, 
               func=q_sample,
               output_dir='./imgs',
               gif_name='gugugaga_denoising.gif',
               reverse=True)
 

GIF saved to: imgs\gugugaga_denoising.gif


去噪的过程差不多就长这个样子:

<div align="center">
    <img src="./imgs/gugugaga_denoising.gif" alt="gugugaga_denoising" style="width: 250px; height: auto;">
</div>


### 损失函数
我们直接从主观上理解一下，既然模型要预测噪声，那损失函数肯定是让预测的噪声和真实的噪声尽可能接近，DDPM 的实现中使用 MSE Loss(均方误差)来衡量预测的质量:

`function loss:`
$$
\mathcal{L}(\theta) = \mathbb{E}_{x_0,\epsilon,t}\left\| \epsilon - \epsilon_\theta(x_t, t) \right\|_2^2
$$

我们可以展开上面的式子，就是用上面的重参数化技巧直接得到加噪t步的图像$x_t$:

`function loss:`
$$
\mathcal{L}(\theta) = \mathbb{E}_{x_0,\epsilon,t}\left\|
\epsilon - \epsilon_\theta\Big(
\sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\;t
\Big)
\right\|_2^2
$$

训练时是这样操作的:

1.随机选取一张图像$x_0$和时间步$t$

2.采样添加的高斯噪声$\epsilon$

3.直接得到加噪图像$x_t$

4.模型预测噪声$\epsilon_\theta(x_t, t)$

5.计算均方误差$\text{MSE}(\epsilon, \epsilon_\theta(x_t, t))$，反向传播算法更新参数

6.循环直到结束(误差小于阈值/循环终止)


## DDPM 最终实现

In [4]:
def gather(consts: torch.Tensor, t: torch.Tensor):
    """Gather consts for $t$ and reshape to feature map shape"""
    c = consts.gather(-1, t)  # 取出t对应的噪声调度、α或α的连乘
    
    # -> [bs, 1, 1, 1]，运算时广播为[bs, C, H, W]
    return c.reshape(-1, 1, 1, 1)  

class DenoiseDiffusion:
    """
    ## Denoise Diffusion
    """
    def __init__(self, eps_model: nn.Module, n_steps: int, device: torch.device):
        """
        * `eps_model` is $\textcolor{lightgreen}{\epsilon_\theta}(x_t, t)$ model
        * `n_steps` is $t$
        * `device` is the device to place constants on
        """
        super().__init__()

        self.device = device
        
        # UNet作为噪声预测模型
        self.eps_model = eps_model

        # 生成噪声调度β
        self.beta = torch.linspace(0.0001, 0.02, n_steps).to(device)

        # α = 1 - β
        self.alpha = 1. - self.beta
        # α的连乘
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)
        # 总加噪步数
        self.n_steps = n_steps
        # σ^2即方差，就是对应的β
        self.sigma2 = self.beta

    def q_xt_x0(self, x0: torch.Tensor, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        #### Get $q(x_t|x_0)$ distribution

        \begin{align}
        q(x_t|x_0) &= \mathcal{N} \Big(x_t; \sqrt{\bar\alpha_t} x_0, (1-\bar\alpha_t) \mathbf{I} \Big)
        \end{align}

        求出x_t的分布的均值和方差
        """

        # [gather](utils.html) $\alpha_t$ and compute $\sqrt{\bar\alpha_t} x_0$
        mean = gather(self.alpha_bar, t) ** 0.5 * x0
        # $(1-\bar\alpha_t) \mathbf{I}$
        var = 1 - gather(self.alpha_bar, t)
        
        return mean, var

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, eps: Optional[torch.Tensor] = None):
        """
        #### Sample from $q(x_t|x_0)$

        \begin{align}
        q(x_t|x_0) &= \mathcal{N} \Big(x_t; \sqrt{\bar\alpha_t} x_0, (1-\bar\alpha_t) \mathbf{I} \Big)
        \end{align}

        求出x_0加噪t步后的图像x_t
        """

        # $\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
        if eps is None:
            eps = torch.randn_like(x0)

        # get $q(x_t|x_0)$
        mean, var = self.q_xt_x0(x0, t)
        
        # Sample from $q(x_t|x_0)$
        return mean + (var ** 0.5) * eps

    def p_sample(self, xt: torch.Tensor, t: torch.Tensor):
        """
        #### Sample from $\textcolor{lightgreen}{p_\theta}(x_{t-1}|x_t)$

        \begin{align}
        \textcolor{lightgreen}{p_\theta}(x_{t-1} | x_t) &= \mathcal{N}\big(x_{t-1};
        \textcolor{lightgreen}{\mu_\theta}(x_t, t), \sigma_t^2 \mathbf{I} \big) \\
        \textcolor{lightgreen}{\mu_\theta}(x_t, t)
          &= \frac{1}{\sqrt{\alpha_t}} \Big(x_t -
            \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\textcolor{lightgreen}{\epsilon_\theta}(x_t, t) \Big)
        \end{align}
        """

        # $\textcolor{lightgreen}{\epsilon_\theta}(x_t, t)$
        eps_theta = self.eps_model(xt, t)
        # [gather](utils.html) $\bar\alpha_t$
        alpha_bar = gather(self.alpha_bar, t)
        # $\alpha_t$
        alpha = gather(self.alpha, t)
        # $\frac{\beta}{\sqrt{1-\bar\alpha_t}}$
        eps_coef = (1 - alpha) / (1 - alpha_bar) ** .5
        # $$\frac{1}{\sqrt{\alpha_t}} \Big(x_t -
        #      \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\textcolor{lightgreen}{\epsilon_\theta}(x_t, t) \Big)$$
        mean = 1 / (alpha ** 0.5) * (xt - eps_coef * eps_theta)
        # $\sigma^2$
        var = gather(self.sigma2, t)

        # $\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
        eps = torch.randn(xt.shape, device=xt.device)
        
        # Sample，根据公式结合模型预测的噪声，求出的均值和方差，之后采样得到x_{t-1}
        return mean + (var ** .5) * eps

    def loss(self, x0: torch.Tensor, noise: Optional[torch.Tensor] = None):
        """
        #### Simplified Loss

        $$L_{\text{simple}}(\theta) = \mathbb{E}_{t,x_0, \epsilon} \Bigg[ \bigg\Vert
        \epsilon - \textcolor{lightgreen}{\epsilon_\theta}(\sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t}\epsilon, t)
        \bigg\Vert^2 \Bigg]$$
        """
        # Get batch size
        batch_size = x0.shape[0]
        # Get random $t$ for each sample in the batch
        # 批梯度下降，一次处理batch_size个样本
        t = torch.randint(0, self.n_steps, (batch_size,), device=x0.device, dtype=torch.long)

        # $\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
        if noise is None:
            noise = torch.randn_like(x0)

        # Sample $x_t$ for $q(x_t|x_0)$
        xt = self.q_sample(x0, t, eps=noise)
        # Get $\textcolor{lightgreen}{\epsilon_\theta}(\sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t}\epsilon, t)$
        eps_theta = self.eps_model(xt, t)

        # MSE loss
        return F.mse_loss(noise, eps_theta)
    
    def sample(self, save_dir, sample_num=1, img_size=64, img_channels=3):
        """采样"""
        self.eps_model.eval()

        with torch.no_grad():
            x = torch.randn([sample_num, img_channels, img_size, img_size], device=self.device)

            for t_ in range(self.n_steps):
                t = self.n_steps - t_ - 1
                x  = self.p_sample(x, x.new_full((sample_num,), t, dtype=torch.long))
            
            # DDPM输入和输出在-1~1之间，在此之前需要转为0~1才能存为图像
            x = (x + 1) / 2.0
            x = x.clamp(0.0, 1.0)

            os.makedirs(save_dir, exist_ok=True)
            for i in range(sample_num):
                save_image(
                    x[i],
                    f"{save_dir}/sample_{i}.png",
                    normalize=False
                )

                print(f"Saved sample_{i}.png to {save_dir}")


In [1]:
# 这里以一张图展示一下训练过程
from c_ddpm import get_config, get_ddpm, DDPM_trainer
import torch


config = get_config('./configs/ddpm_unet_config.yaml')
DDPM_model = get_ddpm(config, False)
trainer = DDPM_trainer(DDPM_model, 
                       torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"),
                       config)

trainer.train()


[Epoch 5] loss: 0.951858
[Epoch 10] loss: 0.907034
[Epoch 15] loss: 0.985367
[Epoch 20] loss: 0.848689
[Epoch 25] loss: 0.770825
[Epoch 30] loss: 0.918570
[Epoch 35] loss: 0.671406
[Epoch 40] loss: 0.605200
[Epoch 45] loss: 0.601009
[Epoch 50] loss: 0.817086
[Epoch 55] loss: 0.547995
[Epoch 60] loss: 0.465840
[Epoch 65] loss: 0.497247
[Epoch 70] loss: 0.422203
[Epoch 75] loss: 0.385762
[Epoch 80] loss: 0.370364
[Epoch 85] loss: 0.335939
[Epoch 90] loss: 0.546997
[Epoch 95] loss: 0.288916
[Epoch 100] loss: 0.283323
[Epoch 105] loss: 0.275741
[Epoch 110] loss: 0.271370
[Epoch 115] loss: 0.440058
[Epoch 120] loss: 0.236606
[Epoch 125] loss: 0.209799
[Epoch 130] loss: 0.202331
[Epoch 135] loss: 0.201305
[Epoch 140] loss: 0.177340
[Epoch 145] loss: 0.375052
[Epoch 150] loss: 0.175163
[Epoch 155] loss: 0.500135
[Epoch 160] loss: 0.158384
[Epoch 165] loss: 0.163078
[Epoch 170] loss: 0.148845
[Epoch 175] loss: 0.150361
[Epoch 180] loss: 0.140451
[Epoch 185] loss: 0.175647
[Epoch 190] loss: 0.1

In [2]:
# 采样示例，因为效果不是很好，所以就不放图了
DDPM_model.sample("./imgs", 10)


Saved sample_0.png to ./imgs
Saved sample_1.png to ./imgs
Saved sample_2.png to ./imgs
Saved sample_3.png to ./imgs
Saved sample_4.png to ./imgs
Saved sample_5.png to ./imgs
Saved sample_6.png to ./imgs
Saved sample_7.png to ./imgs
Saved sample_8.png to ./imgs
Saved sample_9.png to ./imgs


In [4]:
# 或者直接加载checkpoint进行推理，这里使用的是50000iter的ckpt
from c_ddpm import get_config, get_ddpm


config = get_config('./configs/ddpm_unet_config.yaml')
DDPM_model_pretrained = get_ddpm(config, True)
DDPM_model_pretrained.sample("./imgs", 4)


Loaded checkpoint from ckpts\ddmp_unet_iter50000.pth
Saved sample_0.png to ./imgs
Saved sample_1.png to ./imgs
Saved sample_2.png to ./imgs
Saved sample_3.png to ./imgs


展示一下其中一张的效果

<div align="center">
    <img src="./imgs/sample_0.png" alt="sample_0" style="width: 250px; height: auto;">
</div>